<a href="https://colab.research.google.com/github/chaseshallow/ansci_4040_final_class_project/blob/main/Step04bEmbedFull.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 04b · Fingerprint each crop: full hour

Turns **all crops from Step 03b** (~300 crops, a couple of minutes on a T4).

Each masked crop becomes a **fingerprint**: 384 numbers from DINOv3 ViT-S/16 describing coat pattern and shape. Two crops of the same cow should have similar fingerprints (cosine similarity near 1); that is what Re-ID relies on.

**DINOv3 needs access:** request it at huggingface.co/facebook/dinov3-vits16-pretrain-lvd1689m, then add a read token as the Colab secret `HF_TOKEN` (🔑 in the left sidebar, notebook access on). Without access the notebook falls back to the open **DINOv2-small** and says so; `Model.txt` records which one was used.

Writes into the Shared Drive, `CalvingPenPipelineOutputs/<video>/Step04bFull/`, **overwriting** each run:

- `Embeddings.npy` + `EmbeddingIndex.csv`: one fingerprint per crop (`NearDuplicate` marks crops almost identical to the previous one of their track)
- `TrackSimilarity.csv` / `.png`: how alike every pair of tracks is; one cow = one evenly bright block

The folder is shared: one person runs a step at a time. Edit cell 2 and rerun; don't add cells.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub, install the model libraries.
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/ansci_4040_final_class_project.git"
RepoBranch = "main"
RepoDir = "/content/ansci_4040_final_class_project"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
%pip install -q -r "{RepoDir}/CalvingPen/requirements.txt"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline.Embed import EmbedCrops
from CalvingPipeline.Paths import OutputRoot
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

try:
    HfToken = userdata.get("HF_TOKEN")   # from Colab Secrets; never paste the token here
except Exception:
    HfToken = None
print("HF_TOKEN:", "found" if HfToken else "not set - DINOv2 fallback will be used")

In [ ]:
# 2. Settings: the only cell to edit.
VideoStem = "2026-09-26 00-00-00 Calving Pen 2"
VideoOutputDir = OutputRoot / VideoStem   # Shared Drive: CalvingPenPipelineOutputs/<video>/
CropsDir = VideoOutputDir / "Step03bFull"     # step 03 output
OutputDir = VideoOutputDir / "Step04bFull"

ModelId = "facebook/dinov3-vits16-pretrain-lvd1689m"
FallbackModelId = "facebook/dinov2-small"   # open model, used if DINOv3 cannot be loaded

assert (CropsDir / "Crops.csv").exists(), f"Run step 03 first: {CropsDir}"

In [ ]:
# 3. Run: fingerprint every crop, then show how alike the tracks are.
import pandas as pd
from IPython.display import Image, display

Stats = EmbedCrops(CropsDir, OutputDir, VideoStem, ModelId=ModelId, FallbackModelId=FallbackModelId, Token=HfToken)

display(Image(str(OutputDir / "TrackSimilarity.png")))
display(pd.read_csv(OutputDir / "TrackSimilarity.csv", index_col=0))

## About this step

**Model:** **DINOv3 ViT-S/16** (Meta, `facebook/dinov3-vits16-pretrain-lvd1689m`, gated: needs `HF_TOKEN`); fallback **DINOv2-small**. Each crop becomes 384 numbers; cosine similarity compares two crops (≈1 = look the same).

**How to read the heatmap:** each square is the average similarity between the crops of two tracks; bright blocks = tracks that look alike.

**What we found (all 306 crops, one cow):** two blocks by **pose**, not by animal:
- lying: T5, T27, T37, T57, T125, T156, T169 (0.76–0.86 with each other)
- standing / rear / after the birth: T54, T186, T198, T234 (0.74–0.84)
- standing side views T2, T40, T253 sit in between (~0.70–0.78)

Lying vs. standing drops to 0.57–0.62 although it is the same cow. Re-ID must therefore compare a new crop with her **closest** crops, not an average. Different cows need to score below ~0.55; that needs a second cow to check. Track 198 (calf in the mask) still fits with her standing tracks.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.